In [18]:
#Step 1 — Query candidates from database
#Connects to the CHAMPSS database using CandidateViewerQuery, loops over a date range (folders), and collects all candidates matching #specific classifications (<faint>, NEW CANDIDATE) into a single list (all_candidates).
#Step 2 — Run multi-day folding pipeline
#For each candidate, builds arguments and runs multidayfold_pipeline to process data across multiple days; skips candidates already #processed (based on existing output files) and stores results in all_outputs.
#Step 3 — Save results
#Serializes the collected outputs (all_outputs) into a file using pickle, allowing the results to be reloaded later without recomputing.

import sps_databases
import subprocess
from cfbm.bm_data import get_data
import os
import numpy as np
from sps_databases import db_utils, db_api
import scipy
from datetime import datetime, timedelta
from scheduler.run_as_service import run_as_service
import pickle
from sps_pipeline.candidate_viewer import CandidateViewerRegistrar as api

#0-Defining the class which is logs me in to the viewer webstite with more flexibility than fct used in other code
#CandidateViewerQuery and CandidateViewerRegistrar
import requests

class AutomationAPI:
    def __init__(self, base_url, survey_id, username, password):
        self.url = base_url
        self.survey_id = survey_id
        self.username = username
        self.password = password
        self.survey_aval = []
        self.session = requests.Session()

        # Login
        self._call("login", username=username, password=password)
        
        # Register endpoints
        self._register_endpoints()
        
        # Get list of surveys
        self.survey_aval = self._call("get_all_surveys")["surveys"]

        # Check if survey_id is valid
        if survey_id not in self.survey_aval:
            raise ValueError(f"Survey ID {survey_id} is not available. Available surveys: ", self.survey_aval)
        
        # Set survey
        self._call("set_survey", survey=survey_id)

    def _call(self, endpoint, **params):
        response = self.session.post(
            self.url,
            params={"endpoint": endpoint},  # GET
            data=params                      # POST
        )
        response.raise_for_status()
        
        try:
            data = response.json()
            if data["status"] != "success":
                raise RuntimeError(data["message"])
        except ValueError:
            raise RuntimeError("Invalid JSON response", response.text)
        
        return data["data"]

    def _register_endpoints(self):
        for endpoint in self._call("get_endpoints_aval")["endpoints"]:
            name = endpoint["name"]
            param_names = endpoint["params"]

            if hasattr(self, name):
                continue

            def make_method(endpoint_name, endpoint_params):
                def method(self, **kwargs):
                    missing = [p for p in endpoint_params if p not in kwargs]
                    if missing:
                        raise ValueError(f"Missing parameter(s): {', '.join(missing)}")
                    return self._call(endpoint_name, **kwargs)
                method.__name__ = endpoint_name
                method.__doc__  = f"Params: {', '.join(endpoint_params)}"
                return method

            setattr(self.__class__, name, make_method(name, param_names))

api = AutomationAPI(
    "https://sps.chimenet.ca/candidates/index.php?automation", "test", "Viewer Bot", "v4A13BNYwqU5okUZE^h9c&x*blzHrYMi"
)

new_cand_info = api.get_files_by_rating_type(folder="test_21", rating_type="new_candidates")["files"]
print(new_cand_info)
print("-"*40) #just a line to separate candidate
faint_cand_info = api.get_files_by_rating_type(folder="test_21", rating_type="faint")["files"]
print(faint_cand_info)

[{'file': 'Multi_Pointing_Groups_f_25.498_DM_167.483_690ab3fc999bcd1baf1091d9', 'folder': 'test_21', 'remoteUrl': '/candidates/index.php?assets&file=Multi_Pointing_Groups_f_25.498_DM_167.483_690ab3fc999bcd1baf1091d9&folder=test_21&type=candidate_image', 'remoteUrl_alt': '/candidates/index.php?assets&file=Multi_Pointing_Groups_f_25.498_DM_167.483_690ab3fc999bcd1baf1091d9&folder=test_21&type=candidate_image_alt', 'checked': {'status': True, 'result': 'NEW CANDIDATE', 'date': '1775495024', 'by': 'Wenke Xia', 'rater_results': {'Wenke Xia': {'result': 'J2043+31', 'additional': False}, 'Viewer Admin': {'result': '<faint>', 'additional': False}}, 'rating_consistency': {'consistent': False, 'status': 'inconsistent', 'result': '<pending>', 'date': '1775495024'}, 'info': {'history': []}, 'tags': []}}]
----------------------------------------
[{'file': 'Multi_Pointing_Groups_f_28.890_DM_20.240_690b7d1b999bcd1baf29e75b', 'folder': 'test_21', 'remoteUrl': '/candidates/index.php?assets&file=Multi_Po

In [19]:
#Step_1-Query website candidates and put them in a list
from sps_pipeline.candidate_viewer import CandidateViewerQuery
from sps_pipeline.candidate_viewer import CandidateViewerRegistrar
from multiday_search import multidayfold_pipeline
import os

# Database configuration
db_config = {
    'host': 'sps-archiver1',
    'user': 'automation',
    'port': 3306,
    'password': '',#no password for automation user
    'database': 'champss'
}

#Creating a list of date
start_date = datetime(2026, 3, 10)#3,22
end_date   = datetime(2026, 3, 10)#4,8

folders = []
current = start_date

while current <= end_date:
    folders.append(current.strftime("%Y-%m-%d"))
    current += timedelta(days=1)
    
#query = CandidateViewerQuery(survey="stackcands", db_config=db_config)
#candidates = query.get_metadata(folder="stack_0")
classifications = ['<faint>', 'NEW CANDIDATE'
                  ]

all_candidates = []
with CandidateViewerQuery(survey='test', db_config=db_config) as query: 
    for folder in folders:
        print(f"\nProcessing folder: {folder}")

        for cls in classifications:
            try:
                candidates = query.get_ratings(
                    folder="test_21",
                    classification=cls,
                    with_metadata=True
                )
            except Exception:
                print(f"No data for {folder}")
                continue  # skip this classification if query fails

            
            print(f"Found {len(candidates)} candidates for {cls} in {folder}")
            all_candidates.extend(candidates)
            print(all_candidates)

print("Query finished")


Processing folder: 2026-03-10
Found 3 candidates for <faint> in 2026-03-10
[{'file': 'Multi_Pointing_Groups_f_6.160_DM_91.584_69099a28999bcd1baff33f53', 'folder': 'test_21', 'result': '<faint>', 'date': 1769572718, 'modified_by': 'Wenke Xia', 'info': '{"history":[]}', 'id': 47, 'rater_a': 'Wenke Xia', 'result_a': '<none>', 'rater_b': 'Viewer Admin', 'result_b': '<faint>', 'additional_ratings': '{}', 'metadata': {'survey': 'test', 'folder': 'test_21', 'file': 'Multi_Pointing_Groups_f_6.160_DM_91.584_69099a28999bcd1baff33f53', 'input_file': '/mnt/beegfs-client/mp_runs/daily_20251031/candidates/Multi_Pointing_Groups_f_6.160_DM_91.584_69099a28999bcd1baff33f53.npz', 'candidate': '', 'telescope': 'chime', 'epoch_topo': '', 'epoch_bary': '', 't_sample': '', 'data_folded': '', 'data_avg': '', 'data_stdev': '', 'profile_bins': '', 'profile_avg': '', 'profile_stdev': '', 'reduce_chi_sqr': '', 'prob_noise': '12.479217529296877', 'best_dm': '91.58412946195824', 'p_topo': '', 'p_topo_d1': '', 'p_t

In [24]:
#Step_2-Run the multi-day fold
#Taging process:We have two tag done or failed, we run the fold on all cand which have either no tag or failed tag.
#After the folding for those who suceeded we add the done tag and we if it apply remove the failed tag. We put nday=0 so that
#if it was ever folded it won't fold again. We can add/remove a specific tag and can have more than one with no problem
all_outputs = []
for cand in all_candidates:
    metadata = cand['metadata']
    input_file = metadata['input_file']
    outfile = f"/mnt/beegfs-client/processed/multiday/{metadata['file']}"


    file_id = metadata["file"]

    details = api.get_file_details(folder=folder, file=file_id)
    tags = details.get("tags", []) if details else []

    # Skip only if already successful
    if "multidayfold_done" in tags:
        print(f"Skipping {outfile} (already done)")
        continue

    # Otherwise run (this includes: no tag OR failed tag)
    print(f"Running {outfile}")
    
    print(f"\nRunning multidayfold_pipeline for {file_id}...")

    #to run in terminal(need conversion to python file also:)command = f"multidayfold_pipeline --candpath {input_file}
    #--db-name champss_processing --nday 0 --datpath /mnt/beegfs-client/raw/ --foldpath /mnt/beegfs-client/processed/archives/ --use-workflow"
    args = [
    "--candpath",
    input_file,
    "--db-name",
    "champss_processing",
    "--nday",
    "1",   
    "--datpath",
    "/mnt/beegfs-client/raw/",
    "--foldpath",
    "/mnt/beegfs-client/processed/archives/",
    "--use-workflow",
    "--docker-image-name",
    "sps-archiver1.chime:5000/champss_software:run_on_compute1"
]

    #nday=0 is to run over all available days
    #you are starting a service that starts another service. 
    #The first one will be on sps-compute1 if you start it from your branch, but the second not
    #so we put the docker command so that both are on compute1

    #We want/need to add something to avoid rerunning the fold on candidate we did on previous day (the flag!!!)
    # Running the command
    try:
        fold_output = multidayfold_pipeline.main(
        args=args,
        standalone_mode=False
    )
        if fold_output is None:
            print(f"[ERROR] fold_output is None for {file_id}")
            continue


        status = fold_output[0].get("status", None)

        if status != "success":
            api.add_tag(folder=folder, file=file_id, tag="multidayfold_failed")
            print(f"[WARNING] Workflow failed for {file_id}")
            continue

        all_outputs.append([cand, fold_output[0]])


        api.add_tag(
            folder=folder,
            file=file_id,
            tag="multidayfold_done"
        )

        print(f"Fold finished. Output should be at: {outfile}")
        
    except Exception as e:
        print(f"Folding failed for: {file_id}")
        print(e)
        api.add_tag(
            folder=folder,
            file=file_id,
            tag="multidayfold_failed"
        )
        #add workflow option to the command to avoid error

Running /mnt/beegfs-client/processed/multiday/Multi_Pointing_Groups_f_6.160_DM_91.584_69099a28999bcd1baff33f53

Running multidayfold_pipeline for Multi_Pointing_Groups_f_6.160_DM_91.584_69099a28999bcd1baff33f53...
Source md_248.01_45.48_6.160038_91.58 already in the follow-up source database.


06 May 2026 21:17:12 UTC INFO      root Initial buckets entries: []

Initial buckets entries: []
Initial buckets entries: []
Initial buckets entries: []


06 May 2026 21:17:12 UTC INFO      root Final buckets entries: []

Final buckets entries: []
Final buckets entries: []
Final buckets entries: []


...

06 May 2026 21:18:45 UTC INFO      root Removing finished service                                                  
                                  processing-multiday-confirm-69fb643a6109e1d023432a65 in state complete.

Removing finished service processing-multiday-confirm-69fb643a6109e1d023432a65 in state complete.
Removing finished service processing-multiday-confirm-69fb643a6109e1d023432a65 in state complete.
Removing finished service processing-multiday-confirm-69fb643a6109e1d023432a65 in state complete.


06 May 2026 21:18:46 UTC INFO      root Workflow Results for Work ID 69fbb028b69fc15793b80386:                     
                                  [{'pipeline': 'champss-multiday-confirm', 'site': 'chime', 'user': 'CHAMPSS',    
                                  'function': 'multiday_search.confirm_cand.main', 'parameters': {'fs_id':         
                                  '69fb643a6109e1d023432a65', 'db_host': 'sps-archiver1', 'db_port': 27017,        
                                  'db_name': 'champss_processing', 'nday': 1, 'write_to_db': True, 'foldpath':     
                                  '/mnt/beegfs-client/processed/archives/'}, 'command': None, 'results': {'locked':
                                  False}, 'products': None, 'plots': None, 'tags': ['multiday', 'confirm',         
                                  '69fb643a6109e1d023432a65', '69fbb02832a30a5fe4c013a6'], 'event': None, 'id':    
                                  '69fbb028b69fc15793b80386', 'creation': 1778102312.630201, 'start':              
                                  1778102314.8455837, 'stop': 1778102323.231999, 'attempt': 1, 'status': 'failure',
                                  'timeout': 7200, 'retries': 1, 'priority': 3, 'config': {'archive': {'results':  
                                  True, 'products': 'bypass', 'plots': 'bypass', 'logs': 'move'}, 'metrics': False,
                                  'parent': None, 'orgs': ['chimefrb'], 'teams': None}, 'notify': {'slack':        
                                  {'channel_id': None, 'member_ids': None, 'message': None, 'results': None,       
                                  'products': None, 'plots': None, 'blocks': None, 'reply': None}}}]

Workflow Results for Work ID 69fbb028b69fc15793b80386: 
[{'pipeline': 'champss-multiday-confirm', 'site': 'chime', 'user': 'CHAMPSS', 'function': 'multiday_search.confirm_cand.main', 'parameters': {'fs_id': '69fb643a6109e1d023432a65', 'db_host': 'sps-archiver1', 'db_port': 27017, 'db_name': 'champss_processing', 'nday': 1, 'write_to_db': True, 'foldpath': '/mnt/beegfs-client/processed/archives/'}, 'command': None, 'results': {'locked': False}, 'products': None, 'plots': None, 'tags': ['multiday', 'confirm', '69fb643a6109e1d023432a65', '69fbb02832a30a5fe4c013a6'], 'event': None, 'id': '69fbb028b69fc15793b80386', 'creation': 1778102312.630201, 'start': 1778102314.8455837, 'stop': 1778102323.231999, 'attempt': 1, 'status': 'failure', 'timeout': 7200, 'retries': 1, 'priority': 3, 'config': {'archive': {'results': True, 'products': 'bypass', 'plots': 'bypass', 'logs': 'move'}, 'metrics': False, 'parent': None, 'orgs': ['chimefrb'], 'teams': None}, 'notify': {'slack': {'channel_id': None, 'm

Finished multiday search
[WARNING] Workflow failed for Multi_Pointing_Groups_f_25.498_DM_167.483_690ab3fc999bcd1baf1091d9


In [25]:
#Check if it worked

# refresh data
updated = api.get_files_by_rating_type(folder="test_21", rating_type="faint")

for f in updated["files"]:
    for t in f["checked"]["tags"]:
        print(t["tag"])

Test Tag2
Test Tag3
Confirmed Pulsar
Test Tag2
Test Tag3
Test Tag2
Test Tag3
